## Purpose:
    This script generates Basic Markov baseline routes using transition probabilities
    estimated from observed training routes.

## Input:
    - Hexagon road network graph:
      ./new_hexagraph/hexa_network_with_road.gpickle
    - Training route file:
      ./Jeju_data/Training/shortcut_route/gps_shortcut.csv
    - Validation OD route segment files:
      ./Jeju_data/Validation/shortcut_route/route_split_{k}/
      where k = 8, 12, and 16

## Output:
    - Segment-level Basic Markov route predictions:
      ./prediction_markov/markov_route_split_{k}.csv

    - Traveler-level Basic Markov route predictions:
      ./prediction_markov/markov_route_{k}.csv

## Main procedures:
    1. Load the hexagon road network graph.
    2. Initialize transition counts only for graph-allowed edges.
    3. Accumulate observed transitions from training shortcut routes.
    4. Convert transition counts into row-normalized transition probabilities.
    5. Convert transition probabilities into edge costs using -log(p).
    6. Generate the most probable route for each validation OD segment using Dijkstra's algorithm.
    7. Concatenate segment-level predictions into traveler-level predicted routes.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import pickle
from torch_geometric.utils import from_networkx
import torch
import ast
import re
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
with open("./new_hexagraph/hexa_network_with_road.gpickle", 'rb') as f:
    G = pickle.load(f)
data = from_networkx(G)

num_nodes = data.num_nodes  # 0..N-1
data.x = torch.eye(num_nodes)
node_feat_dim = data.x.size(1)

node_features = data.x
edge_index = data.edge_index

In [ ]:
# Step 1: Initialize transition counts and adjacency mask.
epsilon = 1e-6
route_count = np.zeros((num_nodes, num_nodes), dtype=float)
adj_mask    = np.zeros((num_nodes, num_nodes), dtype=float)

# 실제 연결된 edge(0-based)에만 epsilon 부여 & 마스크 설정
for i in range(edge_index.shape[1]):
    u = int(edge_index[0, i].item())  # 0..N-1
    v = int(edge_index[1, i].item())
    route_count[u, v] += epsilon
    adj_mask[u, v] = 1.0

# Step 2: Accumulate observed transitions from training routes.
route_Tr = pd.read_csv('./Jeju_data/Training/shortcut_route/gps_shortcut.csv')

for i in range(len(route_Tr)):
    raw = ast.literal_eval(route_Tr.iloc[i, 1])
    route = [int(x) for x in raw]  # 0..N-1
    for j in range(len(route) - 1):
        s = route[j]
        e = route[j + 1]
        if 0 <= s < num_nodes and 0 <= e < num_nodes:
            route_count[s, e] += 1
        else:
            print(f"[!] out-of-range in Training: s={s}, e={e}")

# Step 3: Normalize transition counts and retain only graph-allowed transitions.
row_sums = route_count.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
route_prob = route_count / row_sums

# Retain only transitions allowed by the road network graph.
route_prob *= adj_mask

# Re-normalize transition probabilities after applying the adjacency mask.
row_sums2 = route_prob.sum(axis=1, keepdims=True)
row_sums2[row_sums2 == 0] = 1.0
route_prob = route_prob / row_sums2

# Step 4: Build a weighted directed graph using -log(p) as edge cost.
G_nx = nx.DiGraph()
tiny = 1e-12
for u in range(num_nodes):
    cols = np.nonzero(route_prob[u])[0]
    for v in cols:
        p = max(float(route_prob[u, v]), tiny)
        G_nx.add_edge(u, v, weight=-np.log(p))

# # Step 5: Define a route generation function using Dijkstra's algorithm on the weighted graph.
def make_route(source_0, target_0):
    try:
        return nx.dijkstra_path(G_nx, source=source_0, target=target_0, weight='weight')
    except nx.NetworkXNoPath:
        return None

## 예측

In [ ]:
def numerical_sort_key(file):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

def flatten(routes):
    return [p for route in routes for p in route]

In [ ]:
# Generate predictions for validation OD route segments.
for k in range(8, 17, 4):
    csv_dir_Va = f'./Jeju_data/Validation/shortcut_route/route_split_{k}'
    route_split_list_Va = sorted(
        [f for f in os.listdir(csv_dir_Va) if f.endswith(".csv")],
        key=numerical_sort_key
    )

    results = []

    for file_name in route_split_list_Va:
        tmp_r = pd.read_csv(os.path.join(csv_dir_Va, file_name))

        # he first and last nodes of each segment are used as origin and destination nodes
        start_idx = int(tmp_r.iloc[0, 0])
        end_idx   = int(tmp_r.iloc[-1, 0])
        route_0b = make_route(start_idx, end_idx)

        travel_name = tmp_r.iloc[0, 1] if tmp_r.shape[1] > 1 else f"travel_{file_name}"
        if route_0b is None:
            results.append({'travel_name': travel_name, 'predicted_route': []})
        else:
            results.append({'travel_name': travel_name, 'predicted_route': route_0b})


    out_dir = './prediction_markov'
    os.makedirs(out_dir, exist_ok=True)

    df_result = pd.DataFrame(results)
    df_result.to_csv(f'{out_dir}/markov_route_split_{k}.csv', index=False)

    grouped_df = df_result.groupby('travel_name')['predicted_route'] \
                          .apply(flatten).reset_index()
    grouped_df.to_csv(f'{out_dir}/markov_route_{k}.csv', index=False)

    print("finish", k)

print("All done.")

finish 8
finish 12
finish 16
All done.
